# <font color='gold'>Sampling Teacher review text</font>


In [1]:
import pandas as pd
import numpy as np
from nltk import word_tokenize
from sklearn.model_selection import StratifiedShuffleSplit
from sampling_util.table_util import display_count_table, compare_source_and_sample  # helper functions to make tables

#===========================================================
# Parameters
#===========================================================
random_state = 73

input_file = '../../data/1_processed/1_He_2020_RMP_v2.csv'

target_sample_size = 3000

rating_bins = [1, 2.5, 4, 5]
rating_labels = {'(0.999, 2.5]':'Neg', '(2.5, 4.0]':'Conf/Neu',  '(4.0, 5.0]':'Pos'}

length_bin_interval_size = 20

output_cols = ['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin']
#===========================================================

updated_sourcefile = input_file.replace('.csv', '_with_sampling_criteria.csv')
output_file_name = input_file.replace('1_processed/', '2_sampled/sampling_output_unlabelled/').replace('.csv', f'(N={target_sample_size}).csv')

rating_stats_file = input_file.replace('1_processed/', '2_sampled/sampling_output_stats/').replace('.csv', f'(N={target_sample_size})_rating_stats.csv')
length_stats_file = input_file.replace('1_processed/', '2_sampled/sampling_output_stats/').replace('.csv', f'(N={target_sample_size})_length_stats.csv')

print(f'\nInput:      {input_file}\n\n'
      f'Source_v2:  \033[032m{updated_sourcefile}\033[0m\n\n'
      f'Output:     \033[033m{output_file_name}\033[0m\n\n'
      f'Stats_1:    \033[35m{rating_stats_file}\033[0m\n'
      f'Stats_2:    \033[35m{length_stats_file}\033[0m\n')



Input:      ../../data/1_processed/1_He_2020_RMP_v2.csv

Source_v2:  ../../data/1_processed/1_He_2020_RMP_v2_with_sampling_criteria.csv

Output:     ../../data/2_sampled/sampling_output_unlabelled/1_He_2020_RMP_v2(N=3000).csv

Stats_1:    ../../data/2_sampled/sampling_output_stats/1_He_2020_RMP_v2(N=3000)_rating_stats.csv
Stats_2:    ../../data/2_sampled/sampling_output_stats/1_He_2020_RMP_v2(N=3000)_length_stats.csv



## <font color='cornflowerblue'>1) Prepare helper columns</font>


In [3]:
source_df = pd.read_csv(input_file)
print(source_df.shape)

# Get review length (as number of tokens)
source_df.loc[:, 'token_list'] = source_df.review_text.str.strip().apply(word_tokenize)
source_df.loc[:, 'num_token'] = source_df.token_list.apply(len)

# Get sampling_feature distribution
print("distinct teacher names: ", len(source_df.teacher_name.unique()))  # too many to use 
display_count_table(source_df, 'num_token', sort_by='num_token')
display_count_table(source_df, 'rating', sort_by='rating', ascending=True)

print(f"\n\033[34m{'='*50}\033[0m\n") #===========================================

# Bin ratings into groups
source_df.loc[:, 'rating_bin'] = pd.cut(source_df.rating, bins=rating_bins, labels=None, include_lowest=True).astype(str) #, labels=None will show the bin value range
source_df.loc[:, 'rating_cat'] = source_df.rating_bin.map(rating_labels)  #or source_df.rating.apply(lambda x: 'Pos' if 4 < x <= 5 else 'Neg' if 1 <= x <= 2.5 else 'Conf/Neu')
display_count_table(source_df, ['rating_cat', 'rating_bin'], sort_by='rating_bin', ascending=True)
print()

# Bin num_token with the given interval size
interval_size = length_bin_interval_size
maxval = max(source_df.num_token.unique())
bin_edges = np.arange(1, maxval + interval_size, interval_size)

source_df.loc[:, 'length_bin'] = pd.cut(source_df.num_token, bins=bin_edges, labels=None, include_lowest=True)
display_count_table(source_df, 'length_bin', sort_by='length_bin', ascending=True)
print()

#=================================================================
# Create combined column for multi-criteria stratified sampling
source_df.loc[:, 'sample_criteria'] = source_df.rating_cat.astype(str) + '_' + source_df.length_bin.astype(str) 

display_count_table(source_df, 'sample_criteria', sort_by='sample_criteria')

source_df.to_csv(updated_sourcefile, index=False)
print(f'\nsource_df written to \033[33m{updated_sourcefile}\033[0m, done~!\n')
#=================================================================

source_df

(19145, 9)
distinct teacher names:  2687


,num_token,count
80,1,58
76,2,129
68,3,148
75,4,129
72,5,134
...,...,...
95,101,1
91,103,1
93,108,1
92,110,1


,rating,count
3,1.0,2053
7,1.5,794
4,2.0,1167
8,2.5,729
5,3.0,1155
6,3.5,961
2,4.0,2316
1,4.5,2456
0,5.0,7514


,rating_cat,rating_bin,count
1,Neg,"(0.999, 2.5]",4743
2,Conf/Neu,"(2.5, 4.0]",4432
0,Pos,"(4.0, 5.0]",9970


,length_bin,count
3,"(0.999, 21.0]",3057
2,"(21.0, 41.0]",4222
1,"(41.0, 61.0]",4949
0,"(61.0, 81.0]",6757
4,"(81.0, 101.0]",156
5,"(101.0, 121.0]",3
6,"(121.0, 141.0]",1


,sample_criteria,count
11,"Conf/Neu_(0.999, 21.0]",551
15,"Conf/Neu_(101.0, 121.0]",2
9,"Conf/Neu_(21.0, 41.0]",896
6,"Conf/Neu_(41.0, 61.0]",1188
5,"Conf/Neu_(61.0, 81.0]",1754
14,"Conf/Neu_(81.0, 101.0]",41
10,"Neg_(0.999, 21.0]",657
8,"Neg_(21.0, 41.0]",920
7,"Neg_(41.0, 61.0]",1188
3,"Neg_(61.0, 81.0]",1934



source_df written to ../../data/1_processed/1_He_2020_RMP_v2_with_sampling_criteria.csv, done~!



,Unnamed: 0,id,index,department,teacher_name,rating,review_text,source,type,token_list,num_token,rating_bin,rating_cat,length_bin,sample_criteria
0,0,1_90_2430_1,0,English As A Second Language,Steven Cunningham,1.0,"Never trust yourself, you would be surprised b...",data1_19145,teacher review,"[Never, trust, yourself, ,, you, would, be, su...",27,"(0.999, 2.5]",Neg,"(21.0, 41.0]","Neg_(21.0, 41.0]"
1,1,1_236_180_2,1,Spanish,Armando Valdes,1.0,"This class is an easy A, but don\'t expect to ...",data1_19145,teacher review,"[This, class, is, an, easy, A, ,, but, don\'t,...",75,"(0.999, 2.5]",Neg,"(61.0, 81.0]","Neg_(61.0, 81.0]"
2,2,1_182_1691_3,2,Music,Marcos Krieger,1.0,Overall terrible experience.,data1_19145,teacher review,"[Overall, terrible, experience, .]",4,"(0.999, 2.5]",Neg,"(0.999, 21.0]","Neg_(0.999, 21.0]"
3,3,1_40_401_4,3,Business,Charles Hurt,1.0,He did not teach me a single thing. I went int...,data1_19145,teacher review,"[He, did, not, teach, me, a, single, thing, .,...",73,"(0.999, 2.5]",Neg,"(61.0, 81.0]","Neg_(61.0, 81.0]"
4,4,1_37_1575_5,4,Biology,Leonardo Kurachi Ube,1.0,Not a bad professor as he is very passionate a...,data1_19145,teacher review,"[Not, a, bad, professor, as, he, is, very, pas...",66,"(0.999, 2.5]",Neg,"(61.0, 81.0]","Neg_(61.0, 81.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19140,19140,1_187_499_19141,19667,Nursing,Crystal Porter,5.0,Took her back in the spring. Best Damn teacher...,data1_19145,teacher review,"[Took, her, back, in, the, spring, ., Best, Da...",76,"(4.0, 5.0]",Pos,"(61.0, 81.0]","Pos_(61.0, 81.0]"
19141,19141,1_45_947_19142,19668,Chemistry,Henry Ajie,5.0,I had 2 semesters class with him. He is trying...,data1_19145,teacher review,"[I, had, 2, semesters, class, with, him, ., He...",57,"(4.0, 5.0]",Pos,"(41.0, 61.0]","Pos_(41.0, 61.0]"
19142,19142,1_45_328_19143,19669,Chemistry,C Davis,5.0,He is a great professor; and every helpful per...,data1_19145,teacher review,"[He, is, a, great, professor, ;, and, every, h...",32,"(4.0, 5.0]",Pos,"(21.0, 41.0]","Pos_(21.0, 41.0]"
19143,19143,1_82_1317_19144,19670,Economics,Joseph Hotz,5.0,Have to say that his problems sets really take...,data1_19145,teacher review,"[Have, to, say, that, his, problems, sets, rea...",44,"(4.0, 5.0]",Pos,"(41.0, 61.0]","Pos_(41.0, 61.0]"


In [3]:
# ==================================================================================
# As sklearn requires bin with count > 1, 
# identify bins with fewer than 2 samples, duplicate that row (to be removed later)
# ==================================================================================
criteria_bin_counts = source_df['sample_criteria'].value_counts() 
big_bin_criteria_vals = criteria_bin_counts[criteria_bin_counts == max(criteria_bin_counts)].index[0]
single_count_bin_criteria_vals = criteria_bin_counts[criteria_bin_counts < 2].index
print("max_count_bin criteria vals:       ", big_bin_criteria_vals)
print("single_count_bin criteria vals::   ", single_count_bin_criteria_vals, '\n')

display_count_table(source_df, 'sample_criteria', sort_by='count', ascending=False) 
print(source_df.shape)

# For bins with only 1 sample, duplicate them so that the sampling module won't throw error. 
# Will drop duplicates after sampling if they exist in the selected samples
for criteria_value in single_count_bin_criteria_vals: 
    source_df = pd.concat([source_df, source_df.loc[source_df.sample_criteria == criteria_value]], ignore_index=True, sort=False)

display_count_table(source_df, 'sample_criteria', sort_by='count', ascending=False)
print(source_df.shape)

max_count_bin criteria vals:        Pos_(61.0, 81.0]
single_count_bin criteria vals::    Index(['Pos_(121.0, 141.0]', 'Pos_(101.0, 121.0]'], dtype='object', name='sample_criteria') 



,sample_criteria,count
0,"Pos_(61.0, 81.0]",3069
1,"Pos_(41.0, 61.0]",2573
2,"Pos_(21.0, 41.0]",2406
3,"Neg_(61.0, 81.0]",1934
4,"Pos_(0.999, 21.0]",1849
5,"Conf/Neu_(61.0, 81.0]",1754
6,"Conf/Neu_(41.0, 61.0]",1188
7,"Neg_(41.0, 61.0]",1188
8,"Neg_(21.0, 41.0]",920
9,"Conf/Neu_(21.0, 41.0]",896


(19145, 15)


,sample_criteria,count
0,"Pos_(61.0, 81.0]",3069
1,"Pos_(41.0, 61.0]",2573
2,"Pos_(21.0, 41.0]",2406
3,"Neg_(61.0, 81.0]",1934
4,"Pos_(0.999, 21.0]",1849
5,"Conf/Neu_(61.0, 81.0]",1754
6,"Conf/Neu_(41.0, 61.0]",1188
7,"Neg_(41.0, 61.0]",1188
8,"Neg_(21.0, 41.0]",920
9,"Conf/Neu_(21.0, 41.0]",896


(19147, 15)


## <font color='cornflowerblue'>2) Extract samples & post-processing</font>


In [4]:
# Get the stratified sample
sss = StratifiedShuffleSplit(n_splits=1, test_size=target_sample_size + len(single_count_bin_criteria_vals), random_state=random_state)
_train_idx, sample_idx = next(sss.split(source_df.copy(), source_df.sample_criteria))  # iterate through n_split, which is 1. Only need the 'sample_idx' variable
 
sample_df = source_df.iloc[sample_idx]
sample_df = sample_df[output_cols]

# now drop the duplicated row added earlier
print("before drop_duplicates:       ", sample_df.shape)
sample_df = sample_df.drop_duplicates(keep='first')
print("after  drop_duplicates:       ", sample_df.shape, '\n')

display_count_table(sample_df, 'sample_criteria', sort_by='count', ascending=False)
print()

# If current sample size > target_sample_size, adjust the sample size 
diff = len(sample_df) - target_sample_size
if diff > 0:  
    # diff > 0 means no duplicated rows were selected in this sample, so the drop_duplicates above made no change
    # In this case, drop same-number rows randomly from the largest sample_criteria category
    print(f"\033[35m\nNo duplicated rows in sample.\n\033[36mRemoving {diff} random-sample of sample_criteria == ({big_bin_criteria_vals}) to match target sample size !!!!!\n\033[0m")
    drop_scope = sample_df.loc[sample_df.sample_criteria == big_bin_criteria_vals]
    drop_row_idx = drop_scope.sample(n=diff).index
    sample_df = sample_df.drop(index=drop_row_idx)
    print(sample_df.shape, '\n')
    display_count_table(sample_df, 'sample_criteria', sort_by='count', ascending=False)
    print()
else:
    print(f"\033[36m\nDuplicated rows in sample already dropped, all good.\n\033[0m")
    
source_df = source_df.drop(columns=['token_list']).drop_duplicates(keep='first')

sample_df

before drop_duplicates:        (3002, 10)
after  drop_duplicates:        (3002, 10) 



,sample_criteria,count
0,"Pos_(61.0, 81.0]",481
1,"Pos_(41.0, 61.0]",404
2,"Pos_(21.0, 41.0]",377
3,"Neg_(61.0, 81.0]",303
4,"Pos_(0.999, 21.0]",290
5,"Conf/Neu_(61.0, 81.0]",275
6,"Conf/Neu_(41.0, 61.0]",186
7,"Neg_(41.0, 61.0]",186
8,"Neg_(21.0, 41.0]",144
9,"Conf/Neu_(21.0, 41.0]",141




No duplicated rows in sample.
Removing 2 random-sample of sample_criteria == (Pos_(61.0, 81.0]) to match target sample size !!!!!

(3000, 10) 



,sample_criteria,count
0,"Pos_(61.0, 81.0]",479
1,"Pos_(41.0, 61.0]",404
2,"Pos_(21.0, 41.0]",377
3,"Neg_(61.0, 81.0]",303
4,"Pos_(0.999, 21.0]",290
5,"Conf/Neu_(61.0, 81.0]",275
6,"Conf/Neu_(41.0, 61.0]",186
7,"Neg_(41.0, 61.0]",186
8,"Neg_(21.0, 41.0]",144
9,"Conf/Neu_(21.0, 41.0]",141


,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
18910,teacher review,data1_19145,1_45_2639_18911,5.0,74,"Pos_(61.0, 81.0]",Dr. Zhong is a really good chemistry teacher. ...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
3439,teacher review,data1_19145,1_206_452_3440,1.5,56,"Neg_(41.0, 61.0]",I have her for behav. neuro and she is terribl...,"(0.999, 2.5]",Neg,"(41.0, 61.0]"
1030,teacher review,data1_19145,1_93_2121_1031,1.0,77,"Neg_(61.0, 81.0]",the prof. is real crap. it is a shame that the...,"(0.999, 2.5]",Neg,"(61.0, 81.0]"
4830,teacher review,data1_19145,1_89_50_4831,3.5,42,"Conf/Neu_(41.0, 61.0]",He is a great guy and is pretty funny in class...,"(2.5, 4.0]",Conf/Neu,"(41.0, 61.0]"
5467,teacher review,data1_19145,1_88_396_5468,1.0,44,"Neg_(41.0, 61.0]","Plain and simple, this guy is HORRIBLE! He not...","(0.999, 2.5]",Neg,"(41.0, 61.0]"
...,...,...,...,...,...,...,...,...,...,...
16257,teacher review,data1_19145,1_37_460_16258,5.0,68,"Pos_(61.0, 81.0]",Dr. Harbison is a great guy. He is very passio...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
7150,teacher review,data1_19145,1_89_2514_7151,4.5,65,"Pos_(61.0, 81.0]",Terry Chance was one of the best teachers I ha...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
13367,teacher review,data1_19145,1_203_111_13368,5.0,60,"Pos_(41.0, 61.0]",A bit intimidating at first but she somehow ma...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
8669,teacher review,data1_19145,1_105_1426_8670,5.0,49,"Pos_(41.0, 61.0]",Keiji is amazing. He is an incredible and acco...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"


## <font color='cornflowerblue'>3) Compare source vs. sample distributions</font>


In [5]:
rating_stats = compare_source_and_sample(source_df, sample_df, ['rating_cat', 'rating_bin'], sort_by='rating_bin', ascending=True).fillna('')
rating_stats.to_csv(rating_stats_file, index=False)
print(f'\nRating_stats written to \033[33m{rating_stats_file}\033[0m, done~!\n')

rating_stats


Rating_stats written to ../../data/2_sampled/sampling_output_stats/1_He_2020_RMP_v2(N=3000)_rating_stats.csv, done~!



,df_name,rating_cat,rating_bin,count,%,df_name,rating_cat,rating_bin,count,%
1,source_df,Neg,"(0.999, 2.5]",4743.0,0.247741,sample_df,Neg,"(0.999, 2.5]",743.0,0.247667
2,source_df,Conf/Neu,"(2.5, 4.0]",4432.0,0.231496,sample_df,Conf/Neu,"(2.5, 4.0]",696.0,0.232000
0,source_df,Pos,"(4.0, 5.0]",9970.0,0.520763,sample_df,Pos,"(4.0, 5.0]",1561.0,0.520333
3,,,,19145.0,1.000000,,,,3000.0,1.000000


In [6]:
length_stats = compare_source_and_sample(source_df, sample_df, ['length_bin'], sort_by='length_bin', ascending=True).fillna('')
length_stats.to_csv(length_stats_file, index=False)
print(f'\nLength_stats written to \033[33m{length_stats_file}\033[0m, done~!\n')

length_stats


Length_stats written to ../../data/2_sampled/sampling_output_stats/1_He_2020_RMP_v2(N=3000)_length_stats.csv, done~!



,df_name,length_bin,count,%,df_name,length_bin,count,%
3,source_df,"(0.999, 21.0]",3057.0,0.159676,sample_df,"(0.999, 21.0]",480.0,0.160000
2,source_df,"(21.0, 41.0]",4222.0,0.220528,sample_df,"(21.0, 41.0]",662.0,0.220667
1,source_df,"(41.0, 61.0]",4949.0,0.258501,sample_df,"(41.0, 61.0]",776.0,0.258667
0,source_df,"(61.0, 81.0]",6757.0,0.352938,sample_df,"(61.0, 81.0]",1057.0,0.352333
4,source_df,"(81.0, 101.0]",156.0,0.008148,sample_df,"(81.0, 101.0]",25.0,0.008333
5,source_df,"(101.0, 121.0]",3.0,0.000157,sample_df,"(101.0, 121.0]",0.0,0.000000
6,source_df,"(121.0, 141.0]",1.0,0.000052,sample_df,"(121.0, 141.0]",0.0,0.000000
7,,,19145.0,1.000000,,,3000.0,1.000000


## <font color='cornflowerblue'>4) Write to file</font>


In [7]:
print(sample_df.shape)
print(list(sample_df.columns.values), '\n')

sample_df.to_csv(output_file_name, index=False)
print(f'sample written to \033[33m{output_file_name}\033[0m, done~!\n')

sample_df

(3000, 10)
['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin'] 

sample written to ../../data/2_sampled/sampling_output_unlabelled/1_He_2020_RMP_v2(N=3000).csv, done~!



,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
18910,teacher review,data1_19145,1_45_2639_18911,5.0,74,"Pos_(61.0, 81.0]",Dr. Zhong is a really good chemistry teacher. ...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
3439,teacher review,data1_19145,1_206_452_3440,1.5,56,"Neg_(41.0, 61.0]",I have her for behav. neuro and she is terribl...,"(0.999, 2.5]",Neg,"(41.0, 61.0]"
1030,teacher review,data1_19145,1_93_2121_1031,1.0,77,"Neg_(61.0, 81.0]",the prof. is real crap. it is a shame that the...,"(0.999, 2.5]",Neg,"(61.0, 81.0]"
4830,teacher review,data1_19145,1_89_50_4831,3.5,42,"Conf/Neu_(41.0, 61.0]",He is a great guy and is pretty funny in class...,"(2.5, 4.0]",Conf/Neu,"(41.0, 61.0]"
5467,teacher review,data1_19145,1_88_396_5468,1.0,44,"Neg_(41.0, 61.0]","Plain and simple, this guy is HORRIBLE! He not...","(0.999, 2.5]",Neg,"(41.0, 61.0]"
...,...,...,...,...,...,...,...,...,...,...
16257,teacher review,data1_19145,1_37_460_16258,5.0,68,"Pos_(61.0, 81.0]",Dr. Harbison is a great guy. He is very passio...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
7150,teacher review,data1_19145,1_89_2514_7151,4.5,65,"Pos_(61.0, 81.0]",Terry Chance was one of the best teachers I ha...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
13367,teacher review,data1_19145,1_203_111_13368,5.0,60,"Pos_(41.0, 61.0]",A bit intimidating at first but she somehow ma...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
8669,teacher review,data1_19145,1_105_1426_8670,5.0,49,"Pos_(41.0, 61.0]",Keiji is amazing. He is an incredible and acco...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
